In [1]:
import pandas as pd
import glob
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import openai
from bertopic.representation import OpenAI

from hdbscan import HDBSCAN

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [9]:
df_news = pd.read_parquet('./Data/all_news_1984_2024.parquet')
df_news['Date'] = pd.to_datetime(df_news['Date'], format='%Y-%m-%d')

In [10]:
year = 2001

df_gpr = pd.read_csv(f'./Result/GPR_Peaks/data/gpr_peaks_{year}.csv')
filtered_news = df_news[df_news['Date'].isin(df_gpr['date'])]
print(filtered_news.sample(5))

docs = filtered_news['Caption'].tolist()

             Date                                            Caption  \
888048 2001-09-18  Initial Investigation Fails to Dig Up Evidence...   
888188 2001-09-19                                        World Watch   
889279 2001-10-01                               Business and Finance   
889425 2001-10-02        Funny Pages Mirror Nation's Reflective Mood   
889176 2001-09-28  Weyerhaeuser Cuts Earnings Outlook, Jobs; Stoc...   

                                                  Content  
888048  Short-selling activity jumped in numerous airl...  
888188  EUROPE Interbrew to Sell Carling Unit in U.K. ...  
889279  AN FASB TASK FORCE FOUND the Sept. 11 terroris...  
889425  NEW YORK -- One gift Brian Walker might cross ...  
889176  Weyerhaeuser Co. scaled back its forecast for ...  


In [11]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")
topic_model = BERTopic(embedding_model=embedding_model, 
                     #   min_topic_size=15,
                      #  nr_topics=5,
                       verbose=False)
topics, probs = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()
print("Topic Info:")
print(topic_info)

Topic Info:
    Topic  Count                                      Name  \
0      -1   1236                           -1_in_of_to_for   
1       0    150                 0_buy_brief_business_corp   
2       1    106          1_anthrax_health_questions_alert   
3       2     87                  2_cut_jobs_layoffs_force   
4       3     75                   3_letters_editor_the_we   
..    ...    ...                                       ...   
91     90     11          90_keefe_bruyette_dealmakers_top   
92     91     11        91_pepsi_coke_bottling_consolidate   
93     92     11                 92_yields_week_mostly_cds   
94     93     11           93_capital_weekly_bureau_street   
95     94     10  94_americas_terrorists_terrorism_carnage   

                                       Representation  \
0   [in, of, to, for, as, on, the, attacks, terror...   
1   [buy, brief, business, corp, unit, sale, wirel...   
2   [anthrax, health, questions, alert, vaccine, b...   
3   [cut, jobs,

In [12]:
# 定义地缘政治风险（GPR）类别描述
gpr_desc = {
    "War Threats": "Mentions of military conflict, invasions, or troop deployments",
    "Peace Threats": "Breakdown of diplomatic efforts, failed peace talks",
    "Military Buildups": "Large-scale troop or arms build-up in border areas",
    "Nuclear Threats": "Risk of nuclear weapon development or deployment",
    "Terror Threats": "Threats, alerts, or warnings about terrorist activity",
    "Beginning of War": "Sudden outbreak of war, declaration of war",
    "Escalation of War": "Intensified fighting or spread of conflict",
    "Terror Acts": "Actual bombings, attacks, hostage situations"
}

# rep_docs = topic_model.get_representative_docs()
# rep_texts = [rep_docs[t][0] if t in rep_docs and rep_docs[t] else "" for t in topic_model.get_topic_info().Topic]
# rep_embed = embedding_model.encode(rep_texts)

topic_embed = topic_model.topic_embeddings_
gpr_embed = [embedding_model.encode(category) for category in gpr_desc.keys()]

gpr_sim = cosine_similarity(topic_embed, gpr_embed)

df_sim_gpr = pd.DataFrame(gpr_sim, columns=gpr_desc.keys())
df_sim_gpr.index = [f'Topic {i}' for i in range(len(topic_embed))]

for gpr_cat in list(gpr_desc.keys()):
    print(df_sim_gpr.sort_values(by=gpr_cat, ascending=False)[[gpr_cat]].head(5))

          War Threats
Topic 19     0.674603
Topic 31     0.574110
Topic 95     0.544601
Topic 0      0.479491
Topic 17     0.463117
          Peace Threats
Topic 95       0.604782
Topic 22       0.546220
Topic 31       0.537454
Topic 19       0.536320
Topic 68       0.489957
          Military Buildups
Topic 19           0.632355
Topic 15           0.435628
Topic 0            0.430708
Topic 31           0.430055
Topic 54           0.376208
          Nuclear Threats
Topic 31         0.452901
Topic 17         0.436971
Topic 19         0.422309
Topic 0          0.408061
Topic 95         0.395407
          Terror Threats
Topic 95        0.696635
Topic 13        0.614117
Topic 41        0.607237
Topic 22        0.586665
Topic 57        0.563901
          Beginning of War
Topic 19          0.590118
Topic 95          0.375140
Topic 31          0.347697
Topic 54          0.339141
Topic 41          0.337487
          Escalation of War
Topic 19           0.605097
Topic 31           0.478009
Topi

In [14]:
gics_desc = {
    "Energy": "The Energy Sector comprises companies engaged in the exploration, production, refining, marketing, storage, and transportation of oil, gas, coal, and consumable fuels, as well as companies that provide equipment and services to oil and gas producers.",
    "Materials": "The Materials Sector includes companies engaged in producing chemicals, construction materials, glass, paper, forest products, and packaging products, as well as metals, minerals, and mining companies.",
    "Industrials": "The Industrials Sector includes companies that produce capital goods used in manufacturing, resource development, and construction. It includes manufacturers and distributors of capital goods, providers of commercial and professional services, and transportation companies.",
    "Consumer Discretionary": "The Consumer Discretionary Sector encompasses industries that are most sensitive to economic cycles. It includes automotive, household durable goods, apparel, and leisure equipment companies. The sector also includes consumer services, such as hotels, restaurants, and leisure facilities, as well as retailing.",
    "Consumer Staples": "The Consumer Staples Sector comprises companies whose businesses are less sensitive to economic cycles. It includes manufacturers and distributors of food, beverages, and tobacco, and non-durable household goods and personal products. It also includes food & staples retailing companies.",
    "Health Care": "The Health Care Sector includes companies in two main industry groups: Health Care Equipment & Services, and Pharmaceuticals, Biotechnology & Life Sciences. It includes companies that manufacture health care equipment and supplies, providers of health care services, and companies involved in the research, development, and production of pharmaceuticals and biotechnology products.",
    "Financials": "The Financials Sector contains companies involved in activities such as banking, diversified financial services, and insurance. This sector includes banks, capital markets companies, insurance companies, and consumer finance companies.",
    "Information Technology": "The Information Technology Sector comprises companies in three main industry groups: Software & Services, Technology Hardware & Equipment, and Semiconductors & Semiconductor Equipment. This includes companies that develop software, provide IT services, and manufacture technology hardware and related components.",
    "Communication Services": "The Communication Services Sector includes companies that provide communication services through fixed-line, cellular, or wireless networks. It also includes media and entertainment companies, such as radio and television broadcasting, and interactive media and services.",
    "Utilities": "The Utilities Sector comprises companies considered electric, gas, or water utilities, or that operate as independent power and renewable electricity producers.",
    "Real Estate": "The Real Estate Sector contains companies engaged in real estate development and operations. It also includes companies that offer real estate-related services and Equity Real Estate Investment Trusts (REITs), excluding Mortgage REITs."
}

topic_embed = topic_model.topic_embeddings_
gics_embed = [embedding_model.encode(category) for category in gics_desc.keys()]

gics_sim = cosine_similarity(topic_embed, gics_embed)

df_sim_gics = pd.DataFrame(gics_sim, columns=gics_desc.keys())
df_sim_gics.index = [f'Topic {i}' for i in range(len(topic_embed))]

for gics_cat in list(gics_desc.keys()):
    print(df_sim_gics.sort_values(by=gics_cat, ascending=False)[[gics_cat]].head(5))

            Energy
Topic 74  0.265495
Topic 39  0.257902
Topic 19  0.256877
Topic 54  0.256346
Topic 18  0.250933
          Materials
Topic 39   0.345047
Topic 74   0.277862
Topic 87   0.275261
Topic 11   0.228244
Topic 34   0.207328
          Industrials
Topic 11     0.454144
Topic 34     0.430218
Topic 74     0.393106
Topic 48     0.362855
Topic 0      0.354657
          Consumer Discretionary
Topic 34                0.428642
Topic 14                0.418420
Topic 79                0.403057
Topic 11                0.350882
Topic 55                0.345547
          Consumer Staples
Topic 11          0.465891
Topic 14          0.348279
Topic 74          0.330772
Topic 34          0.324710
Topic 60          0.272634
          Health Care
Topic 34     0.396175
Topic 76     0.364063
Topic 75     0.364014
Topic 79     0.363317
Topic 54     0.349646
          Financials
Topic 34    0.736629
Topic 11    0.495435
Topic 94    0.461142
Topic 0     0.438831
Topic 40    0.430866
          Inform

In [15]:
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:2])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:2])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

In [16]:
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")

representation_model = OpenAI(
    client, model='gpt-4o' ,exponential_backoff=True, chat=True, prompt=prompt
)

topic_model.update_topics(docs, representation_model=representation_model)

In [17]:
df_result = topic_model.get_topic_info()
# drop Representative_Docs
df_result = df_result.drop(columns=['Representative_Docs'], errors='ignore')
# Save the result to a CSV file
df_result.to_csv(f'./result_{year}_v1.csv')
df_result

,Topic,Count,Name,Representation
0,-1,1236,-1_Economic and Security Impact of Terror Attacks,[Economic and Security Impact of Terror Attacks]
1,0,150,0_Business Acquisitions and Agreements,[Business Acquisitions and Agreements]
2,1,106,1_Anthrax and Bioterrorism Threats,[Anthrax and Bioterrorism Threats]
3,2,87,2_Corporate Layoffs and Workforce Reductions,[Corporate Layoffs and Workforce Reductions]
4,3,75,3_Responses to National Crisis Events,[Responses to National Crisis Events]
...,...,...,...,...
91,90,11,90_Financial Industry Rebuilding and Leadershi...,[Financial Industry Rebuilding and Leadership ...
92,91,11,91_Competitive Strategies and Financial Moves ...,[Competitive Strategies and Financial Moves in...
93,92,11,92_CD Yields Weekly Decline,[CD Yields Weekly Decline]
94,93,11,93_Economic Reports from The Wall Street Journ...,[Economic Reports from The Wall Street Journal...
